In [ ]:
import glob
from dotenv import load_dotenv
from analysis.log_fqe_values import get_fqe_values
from dataset.pre_processing import get_terminated
from dataset.load import load_dataset_to_buffer
import pandas as pd
from dataset.pre_processing_configs import load_pre_processing_configs
from os.path import join
from utils.files import load_json
import seaborn as sns
import itertools
import matplotlib.pyplot as plt
from os.path import getctime

In [ ]:



from dataset.config import DatasetLoadConfig
from utils.files import load_yaml

sns.set_style("whitegrid")
sns.set_context("paper")  # Alternatives: "notebook", "talk", "poster"

# Use a color-blind-friendly palette
sns.set_palette("colorblind")


def load_fqe_mode_paths(exp_path, checkpoint='400000'):
    fqe_model_files = glob.glob(join(exp_path, 'eval', 'Dist-fqe', '**', checkpoint, '*', '*.pkl'), recursive=True)
    print(fqe_model_files)
    fqe_model_files.sort(key=getctime)
    return fqe_model_files[0]


def is_finished(exp):
    return load_json(join(exp, 'experiment_config.json'))['finished']


def fix_path_prefix_and_load_dataset_config(config_path, remove_prefix):
    dataset_config_dict = load_yaml(config_path)

    dataset_config_dict['dataset_path'] = dataset_config_dict['dataset_path'].replace(remove_prefix, '')
    dataset_config_dict['train_split_path'] = dataset_config_dict['train_split_path'].replace(remove_prefix, '')
    dataset_config_dict['test_split_path'] = dataset_config_dict['test_split_path'].replace(remove_prefix, '')
    dataset_config_dict['dataset_normalization_param_path'] = dataset_config_dict[
        'dataset_normalization_param_path'].replace(remove_prefix, '')
    dataset_config_dict['discrete_actions_file_path'] = dataset_config_dict['discrete_actions_file_path'].replace(
        remove_prefix, '')

    return DatasetLoadConfig(**dataset_config_dict)


load_dotenv()

reward_tune_experiment_paths = [

    
    'logs/reward-tune-Hybrid-IQL-2025-07-17 16:20:26.277111',
    'logs/reward-tune-Hybrid-IQL-2025-07-23 17:06:23.219054'
]

experiment_paths = [glob.glob(join(path, 'tasks', '*')) for path in reward_tune_experiment_paths]
experiment_paths = list(itertools.chain(*experiment_paths))


experiment_paths = [exp for exp in experiment_paths if is_finished(exp)]
experiments = [
                  
    'logs/e2e-Hybrid-IQL-2025-07-22 14:55:07.499972/Hybrid-IQL-2025-07-22 14:55:06.989390',
    'logs/e2e-Hybrid-IQL-2025-07-22 18:19:03.782413/Hybrid-IQL-2025-07-22 18:19:03.172563'
              ] + experiment_paths
fqe_model_paths = []


for experiment in experiments:
    fqe_model_paths.append(load_fqe_mode_paths(experiment))

dataset_config_path = join(experiments[0], 'dataset_config.json')
remove_prefix = '/data/horse/ws/muyo366f-intellilung/'

dataset_config = fix_path_prefix_and_load_dataset_config(config_path=dataset_config_path, remove_prefix=remove_prefix)
pre_process_configs = load_pre_processing_configs(
    pre_process_configs_path=join(experiments[0], 'pre_processing_metadata.json'))
dataset = dataset_config.dataset

print(len(experiments))




In [ ]:


patient_alive_col = pre_process_configs.patient_alive_col
ep_id_col = pre_process_configs.episode_id_column
reward_col = 'reward'
mortality_col = 'morta'
total_rew_col = 'total_reward'

time_step_col = pre_process_configs.timestep_column
terminated = get_terminated(dataset=dataset, id_column=ep_id_col)
time_steps = dataset.groupby(ep_id_col)[time_step_col].cumcount().values
epi_id = dataset[ep_id_col].values

q_value_df = dataset[[ep_id_col, time_step_col]]
q_value_df[mortality_col] = 1 - dataset[patient_alive_col]
buffer = load_dataset_to_buffer(dataset=dataset, dataset_configs=dataset_config,
                                pre_process_configs=pre_process_configs, device='cuda')

In [ ]:
from reward.range import RangeReward
from reward.ventilator_free_days import VFDEachStep

range_reward_fn = RangeReward(ranges_file_path='configs/state_vector_ranges_for_reward.json', normalize=True,
                              time_penalty=True)
q_value_df['range_reward'] = range_reward_fn(dataset=dataset, terminated=terminated,
                                             pre_process_configs=pre_process_configs)
q_value_df['vfd_reward'] = VFDEachStep(min_reward=0, max_reward=1)(dataset=dataset, terminated=terminated)
q_value_df['daemo_discharge'] = dataset['daemo_discharge']

In [ ]:
from reward.ventilator_free_days import VFDEachStep, VentilatorFreeReward
reward_type_names = {'VentilatorFreeReward': 'VFD@TerminalStep', 'VFDEachStep': 'VFD@EachStep', 'MortalityReward': 'Mortality'}

grouped_dfs = []
for i, (experiment, fqe_path) in enumerate(zip(experiments, fqe_model_paths)):
    exp_pre_config = load_pre_processing_configs(
        pre_process_configs_path=join(experiment, 'pre_processing_metadata.json'))
    dataset_config = fix_path_prefix_and_load_dataset_config(config_path=join(experiment, 'dataset_config.json'),
                                                             remove_prefix=remove_prefix)

    buffer = load_dataset_to_buffer(dataset=dataset_config.dataset, dataset_configs=dataset_config,
                                    pre_process_configs=exp_pre_config, device='cuda', flatten_hybrid_actions=True)

    fqe_values = get_fqe_values(states=buffer.observations, actions=buffer.actions, fqe_model_path=fqe_path,
                                mini_batch_size=4096)
    df = q_value_df.copy(deep=True)
    df['q_value'] = fqe_values
    algo_config = load_json(join(experiment, 'config.json'))

    total_rewards = len(exp_pre_config.reward_function.reward_fns)
    

 
   
    df['morta_only'] = 0
    long_term_reward = exp_pre_config.reward_function.reward_fns[1]
    reward_fn_type = type(long_term_reward)
    time_penalty = exp_pre_config.reward_function.reward_fns[0].time_penalty
    if reward_fn_type == VFDEachStep:

        #r_range = f'{long_term_reward.min_reward},{long_term_reward.max_reward}'
        scale = long_term_reward.max_reward - long_term_reward.min_reward
    elif reward_fn_type == VentilatorFreeReward:
        #r_range = f'0,{long_term_reward.scale}'
        scale = long_term_reward.scale
    else:
        #r_range = f'0,{long_term_reward.morta_reward_scale}'
        scale = long_term_reward.morta_reward_scale

   
    grouped_df = df.groupby(ep_id_col).agg(
        ep_length=('q_value', 'size'),
        q_mean=('q_value', 'mean'),
        range_reward_mean=('range_reward', 'mean'),
        vfd_reward_mean=('vfd_reward', 'mean'),
        daemo_discharge=('daemo_discharge', lambda x: x.iloc[0])
    ).reset_index()
    reward_type = exp_pre_config.reward_function.reward_fns[1].__class__.__name__
    grouped_df['Reward Type'] = reward_type_names[reward_type]

    #grouped_df['range'] = r_range
    grouped_df['time_penalty'] = time_penalty
    grouped_df['algo'] = algo_config['name']
    grouped_df['reward_scale'] = scale
    grouped_dfs.append(grouped_df)
    print('Finished {}'.format(i+1))


q_value_grouped_stats = pd.concat(grouped_dfs, ignore_index=True)



In [ ]:
ep_length_corr = q_value_grouped_stats.groupby([ 'Reward Type', 'reward_scale', 'time_penalty'])[
                     ['ep_length', 'q_mean']].corr('spearman').iloc[0::2, -1].reset_index()
ep_length_corr = ep_length_corr.rename(columns={'q_mean': 'correlation_episode_length'})
#sns.barplot(ep_length_corr, x='q_mean', y='ep_length', hue='Reward Type')
ep_length_corr

In [ ]:
rr_corr = q_value_grouped_stats.groupby([ 'Reward Type', 'reward_scale', 'time_penalty'])[['range_reward_mean', 'q_mean', ]].corr(
    'spearman').iloc[0::2, -1].reset_index()

rr_corr = rr_corr.rename(columns={'q_mean': 'Corr@RangeReward'})
rr_corr

In [ ]:
q_value_grouped_stats['morta'] = 1 - q_value_grouped_stats['daemo_discharge']
morta_corr = q_value_grouped_stats.groupby([ 'Reward Type', 'reward_scale', 'time_penalty'])[['morta', 'q_mean']].corr(
    'spearman').iloc[0::2, -1].reset_index()
morta_corr = morta_corr.rename(columns={'q_mean': 'Corr@Mortality'})
morta_corr

In [ ]:
vfd_corr = q_value_grouped_stats.groupby([ 'Reward Type', 'reward_scale', 'time_penalty'])[['vfd_reward_mean', 'q_mean', ]].corr(
    'spearman').iloc[0::2, -1].reset_index()

vfd_corr = vfd_corr.rename(columns={'q_mean': 'Corr@VFD'})
vfd_corr

In [ ]:
key_cols = ["Reward Type", "reward_scale", "time_penalty"]     # the shared design factors

df = (
    ep_length_corr[key_cols + ["correlation_episode_length"]]
    .merge(
        rr_corr[key_cols + ["Corr@RangeReward"]],
        on=key_cols,
        how="inner",
    ).merge(
        vfd_corr[key_cols + ["Corr@VFD"]],
        on=key_cols,
        how="inner",
    ).merge(
        morta_corr[key_cols + ["Corr@Mortality"]],
        on=key_cols,
        how="inner",
    )
)

# optional: build a composite score so you can sort and
# highlight the best row in a single line

In [ ]:
mask = (
    (df['Reward Type'].eq('Mortality')  &  ~df['time_penalty']) |   # Mortality + False
    (df['Reward Type'].ne('Mortality') &   df['time_penalty'])      # others   + True
)

df_masked = df[mask]
df_masked

In [ ]:



def pick_tchebycheff(df, to_minimise, to_maximise):
    # build "+" direction
    objs = [-df[c] if c in to_minimise else df[c] for c in to_minimise + to_maximise]
    objs = pd.concat(objs, axis=1)
    ideal, nadir = objs.max(), objs.min()
    rng = ideal - nadir
    score = ((ideal - objs) / rng).max(axis=1)
    df['tchebycheff_score'] = score
    return df




In [ ]:
pick_tchebycheff(df=df_masked, to_minimise=[], to_maximise=['Corr@RangeReward', 'Corr@VFD'])

In [ ]:

import string

sns.set_theme(style="white", rc={"axes.grid": False}, font_scale=1.2)
plt.rcParams["axes.grid"] = False
def plot_correlations_seaborn(
        df,
        corr_cols=('tchebycheff_score',  "Corr@VFD", "Corr@RangeReward"),
        x_col="reward_scale",
        hue_col="Reward Type",
        wrap=None,
        height=4,
        aspect=1.2
    ):
    # Pretty names for the facet titles
    title_map = {
        "tchebycheff_score": "Tchebycheff Scalarization",
        "Corr@RangeReward": "Corr@RangeReward",
        "Corr@VFD": "Corr@VFD",
    }

    # What to put on each facet’s y-axis
    ylabel_map = {
        "Tchebycheff Scalarization": "Tchebycheff Value",
        "Corr@RangeReward": "Correlation",
        "Corr@VFD": "Correlation",
    }

    long = (
        df
        .melt(id_vars=[x_col, hue_col],
              value_vars=corr_cols,
              var_name="correlation_raw",
              value_name="value")
        .assign(correlation=lambda d: d["correlation_raw"].map(title_map))
        .sort_values(x_col)
    )

    g = sns.relplot(
        data=long,
        kind="line",
        x=x_col,
        y="value",
        hue=hue_col,
        col="correlation",
        col_wrap=wrap,
        height=height,
        aspect=aspect,
        marker="o",
        facet_kws=dict(sharey=False)   # <- ensures every facet shows its own y-label
    )

    g.set_titles(col_template="{col_name}")
    letters = string.ascii_lowercase
    for i, (name, ax) in enumerate(zip(g.col_names, g.axes.flat)):
        ax.axhline(0, lw=0.5, ls="--", c="gray")
        ax.grid(True, linewidth=0.3, alpha=0.4)
        ax.set_xlabel("Reward Scale")
        ax.set_ylabel(ylabel_map.get(name, "Value"))
        ax.tick_params(labelleft=True)  # make sure ticks are visible
        ax.grid(False) 
        ax.set_ylim(-0.1, 1)
        ax.annotate(f"({string.ascii_lowercase[i]})",
                xy=(0.5, -0.30), xycoords="axes fraction",  # y < 0 pushes it below
                ha="center", va="top", fontsize=12, fontweight="bold")

    g.fig.tight_layout()
    return g


In [ ]:

plot_correlations_seaborn(df_masked)

In [ ]:
opposite_mask = (
    (df['Reward Type'].eq('Mortality')  &  df['time_penalty']) |   # Mortality + False
    (df['Reward Type'].ne('Mortality') &   ~df['time_penalty'])      # others   + True
)
df_opp_mask = df[opposite_mask]
df_opp_mask

In [ ]:
pick_tchebycheff(df=df_opp_mask, to_minimise=['Corr@Mortality'], to_maximise=['Corr@RangeReward', 'Corr@VFD'])


In [ ]:
plot_correlations_seaborn(df_opp_mask)
